# Start

In [ ]:
import os
import sys
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

In [ ]:
from utils import (
    carrier_renaming,
    gas_boiler,
    get_condense_sum,
    heat_pump,
    resistive_heater,
)

# Add the scripts/pypsa-de directory to the path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), "scripts", "pypsa-de")
sys.path.append(scripts_path)

from flexibility_utils import tech_colors

solar = ["solar", "solar-hsat"]
offwind = ["offwind-ac", "offwind-dc"]
h2_ocgt = ["H2 OCGT", "H2 retrofit OCGT"]

c1_groups = [resistive_heater, gas_boiler, heat_pump, solar, offwind, h2_ocgt]
c1_groups_name = [
    "resistive heater",
    "gas boiler",
    "heat pump",
    "solar",
    "offwind",
    "H2 OCGT",
]


carrier_renaming = {
    "urban central solid biomass CHP CC": "biomass CHP CC",
    "urban central solid biomass CHP": "biomass CHP",
    "urban central gas CHP": "gas CHP",
    "urban central gas CHP CC": "gas CHP CC",
    "urban central coal CHP": "coal CHP",
    "urban central lignite CHP": "lignite CHP",
    "urban central oil CHP": "oil CHP",
    "urban central air heat pump": "air heat pump",
    "urban central resistive heater": "resistive heater",
}

tech_colors["VRES"] = "lightgreen"
tech_colors["H2 OCGT"] = tech_colors["H2"]
tech_colors["H2 retrofit OCGT"] = tech_colors["H2"]
tech_colors["urban central H2 retrofit CHP"] = "turquoise"
tech_colors["battery discharger"] = "darkgoldenrod"
tech_colors["urban central H2 retrofit OCGT"] = "seagreen"
tech_colors["biogas"] = "darkorange"
tech_colors["urban central gas CHP"] = "red"
tech_colors["urban central gas CHP CC"] = "red"
tech_colors["coal"] = "black"
tech_colors["lignite"] = "saddlebrown"
tech_colors["load-shedding"] = "slategrey"
tech_colors["CCGT"] = "lime"
tech_colors["urban central solid biomass CHP"] = "darkgreen"
tech_colors["solid biomass"] = "darkgreen"
tech_colors["urban central coal CHP"] = "black"
tech_colors["urban central H2 CHP"] = tech_colors["H2 OCGT"]
tech_colors["urban central CHP"] = "red"
tech_colors["urban central CHP CC"] = "red"
tech_colors["battery charger"] = "goldenrod"
tech_colors["urban central air heat pump"] = "#8B0000"  # Dark Red
tech_colors["urban decentral air heat pump"] = "#DC143C"  # Crimson
tech_colors["rural ground heat pump"] = "#FF6347"  # Tomato
tech_colors["urban central resistive heater"] = "#006400"  # Dark Green
tech_colors["urban decentral resistive heater"] = "#32CD32"  # Lime Green
tech_colors["rural resistive heater"] = "#3CB371"  # Medium Sea Green
tech_colors["offwind"] = tech_colors["offwind-ac"]
tech_colors["heat pump"] = tech_colors["urban central air heat pump"]
tech_colors["nuclear"] = "#ff8c00"
tech_colors["coal CHP"] = "#8d5e56"
tech_colors["lignite CHP"] = "#8d5e56"
tech_colors["biomass CHP"] = "sandybrown"
tech_colors["oil CHP"] = "#c9c9c9"
tech_colors["load-shedding"] = "pink"
tech_colors["CO2 removal service"] = "orange"


for k, v in carrier_renaming.items():
    if k in tech_colors:
        tech_colors[v] = tech_colors[k]

new_entries = {}
for k, v in tech_colors.items():
    if "urban central " in k:
        new_key = k.replace("urban central ", "")
        new_entries[new_key] = v

tech_colors.update(new_entries)

# replace empty values with grey
tech_colors = {k: ("grey" if v == "" else v) for k, v in tech_colors.items()}

region = "DE"
kwargs = {
    "groupby": pypsa.statistics.groupers["bus", "carrier"],
    "nice_names": False,
}

# Functions

In [ ]:
def plot_balance(nb, title="title", tech_colors=None):
    resample = "D"
    nb = nb.resample(resample).mean()

    df = nb

    # split into df with positive and negative values
    df_neg, df_pos = df.clip(upper=0), df.clip(lower=0)
    df_pos = df_pos[df_pos.sum().sort_values(ascending=False).index]
    df_neg = df_neg[df_neg.sum().sort_values().index]
    # get colors
    c_neg = [
        tech_colors[col] if col in tech_colors else "grey" for col in df_neg.columns
    ]
    c_pos = [
        tech_colors[col] if col in tech_colors else "grey" for col in df_pos.columns
    ]

    fig, ax = plt.subplots(figsize=(14, 8))

    # plot positive values
    ax = df_pos.plot.area(ax=ax, stacked=True, color=c_pos, linewidth=0.0)

    # rename negative values that are also present on positive side, so that they are not shown and plot negative values
    def f(c):
        return "out_" + c

    cols = [f(c) if (c in df_pos.columns) else c for c in df_neg.columns]
    cols_map = dict(zip(df_neg.columns, cols))
    ax = df_neg.rename(columns=cols_map).plot.area(
        ax=ax, stacked=True, color=c_neg, linewidth=0.0
    )

    # explicitly filter out duplicate labels
    handles, labels = ax.get_legend_handles_labels()
    filtered_handles_labels = [
        (h, l) for h, l in zip(handles, labels) if not l.startswith("out_")
    ]
    handles, labels = zip(*filtered_handles_labels)

    # rescale the y-axis
    ax.set_ylim([1.05 * df_neg.sum(axis=1).min(), 1.05 * df_pos.sum(axis=1).max()])
    ax.legend(
        handles,
        labels,
        ncol=1,
        loc="upper center",
        bbox_to_anchor=(1.13, 1.01),
    )
    ax.set_title(title)
    ax.grid(True)

# Loading data

In [ ]:
years = [2025, 2035, 2045]
base_path = "/home/julian-geis/Documents/06_PhD/02Flexibility/runs/"
run = "20251103-flex-4scenarios-27cl-1H"

# Specify which scenarios to load (None = all)
scenarios_to_load = [
    "LowFlexHeatModelMargin0.1",
    "LowBatt50",
    "MedFlex",
    "HigFlex",
    "Ariadne",
]

# Scenario configurations
scenario_configs = {
    "MedFlex": {},
    "HigFlex": {},
    "LowFlexHeatModel": {},
    "LowFlexHeatModelMargin0.1": {},
    "LowFlexHeatPMINPU": {},
    "LowBatt50": {},
    "Ariadne": {
        "path": "/home/julian-geis/Documents/04_Ariadne/run_results/20250214-reworkimportban/KN2045_Bal_v4/postnetworks/base_s_49_lvopt__none_{year}.nc"
    },
}

# Load networks
networks = {}
scenarios = scenarios_to_load if scenarios_to_load else list(scenario_configs.keys())

for scenario in scenarios:
    if scenario not in scenario_configs:
        print(f"Warning: Scenario '{scenario}' not found")
        continue

    networks[scenario] = {}

    # Use custom path or default path
    if "path" in scenario_configs[scenario]:
        path_template = scenario_configs[scenario]["path"]
    else:
        path_template = (
            f"{base_path}/{run}/{scenario}/networks/base_s_27__none_{{year}}.nc"
        )

    for year in years:
        networks[scenario][year] = pypsa.Network(path_template.format(year=year))

# For backward compatibility (optional)
n_mf = networks.get("MedFlex", {})
n_hf = networks.get("HigFlex", {})
n_lfhm = networks.get("LowFlexHeatModel", {})
n_lfhmm = networks.get("LowFlexHeatModelMargin0.1", {})
n_lfhpmpu = networks.get("LowFlexHeatPMINPU", {})
n_ariadne = networks.get("Ariadne", {})

compare_scenarios = list(networks.keys())

In [ ]:
PLOT_DIR = f"/home/julian-geis/Documents/06_PhD/02Flexibility/runs/{run}/system_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

# Capacities

In [ ]:
# capacity elec

region = "DE"
xlims = {
    2020: (-100, 100),
    2025: (-100, 100),
    2030: (-200, 200),
    2035: (-300, 300),
    2040: (-400, 400),
    2045: (-600, 700),
}

for i, year in enumerate(years):
    fig, ax = plt.subplots(1, 1, figsize=(10, 15))

    bc = [
        "AC",
        "low voltage",
    ]  # ['urban central heat', 'rural heat', 'urban decentral heat']

    capas = {}
    for scenario in compare_scenarios:
        capas[scenario] = (
            networks[scenario][year]
            .statistics.optimal_capacity(bus_carrier=bc, **kwargs)
            .filter(like=region)
            .groupby("carrier")
            .sum()
            .div(1e3)
        )

    df = pd.concat([capas[scenario] for scenario in compare_scenarios], axis=1)
    df.columns = compare_scenarios
    df.plot(
        kind="barh",
        title=f"Optimal capacity in {year} ({bc})",
        xlim=xlims[year],
        ylabel="GW",
        ax=ax,
    )

    plt.grid()
    plt.tight_layout()

In [ ]:
# 2045
capas["MedFlex"].filter(like="solar").sum()

In [ ]:
capas["Ariadne"].filter(like="solar").sum()

In [ ]:
capas["MedFlex"].filter(like="wind")  # .sum()

In [ ]:
capas["Ariadne"].filter(like="wind")  # .sum()

In [ ]:
# ratio solar/wind
capas["MedFlex"].filter(like="solar").sum() / capas["MedFlex"].filter(like="wind").sum()

In [ ]:
capas["Ariadne"].filter(like="solar").sum() / capas["Ariadne"].filter(like="wind").sum()

## Generation and Consumption

In [ ]:
# Define carrier groups and carriers to drop
carrier_groups = {
    "electricity": {
        "bus_carrier": ["AC", "low voltage"],
        "drop_carriers": ["AC", "DC"],
    },
    "heat": {
        "bus_carrier": ["urban decentral heat", "rural heat", "urban central heat"],
        "drop_carriers": [],
    },
    "H2": {
        "bus_carrier": ["H2"],
        "drop_carriers": [
            "H2 pipeline",
            "H2 pipeline (Kernnetz)",
            "H2 pipeline retrofitted",
        ],
    },
    "oil": {"bus_carrier": ["oil"], "drop_carriers": []},
    "gas": {"bus_carrier": ["gas"], "drop_carriers": []},
    "co2": {"bus_carrier": ["co2"], "drop_carriers": []},
    "solid_biomass": {"bus_carrier": ["solid biomass"], "drop_carriers": []},
}

# Get results for all carriers
results_by_country = {}
results_by_country_carrier = {}

for carrier_name, config in carrier_groups.items():
    # By country
    temp = {}
    for scenario in compare_scenarios:
        supply = (
            networks[scenario][year]
            .statistics.supply(bus_carrier=config["bus_carrier"], **kwargs)
            .drop(["Store"], errors="ignore")
        )
        df = supply.reset_index()
        df["country"] = df["bus"].str[:2]
        if config["drop_carriers"]:
            df = df[~df["carrier"].isin(config["drop_carriers"])]
        temp[scenario] = df.groupby("country")[0].sum().div(1e6)
    results_by_country[carrier_name] = pd.concat(temp, axis=1)
    results_by_country[carrier_name].columns = compare_scenarios

    # By country and carrier
    temp = {}
    for scenario in compare_scenarios:
        supply = (
            networks[scenario][year]
            .statistics.supply(bus_carrier=config["bus_carrier"], **kwargs)
            .drop(["Store"], errors="ignore")
        )
        df = supply.reset_index()
        df["country"] = df["bus"].str[:2]
        if config["drop_carriers"]:
            df = df[~df["carrier"].isin(config["drop_carriers"])]
        temp[scenario] = df.groupby(["country", "carrier"])[0].sum().div(1e6)
    results_by_country_carrier[carrier_name] = pd.concat(temp, axis=1)
    results_by_country_carrier[carrier_name].columns = compare_scenarios

In [ ]:
year = 2045

### elec

In [ ]:
elec_gen = {}
for scenario in compare_scenarios:
    elec_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["AC", "low voltage"], **kwargs)
        .filter(like="DE")
        .drop(["Store"], errors="ignore")
        .groupby(["carrier"])
        .sum()
        .drop(["AC", "DC"], errors="ignore")
        .div(1e6)
    )  # TWh

elec_gen_df = pd.concat([elec_gen[scenario] for scenario in compare_scenarios], axis=1)
elec_gen_df.columns = compare_scenarios
elec_gen_df[elec_gen_df.gt(1).any(axis=1)]

In [ ]:
# elec consumption
elec_con = {}
for scenario in compare_scenarios:
    elec_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["AC", "low voltage"], **kwargs)
        .filter(like="DE")
        .drop(["Store"], errors="ignore")
        .groupby(["carrier"])
        .sum()
        .drop(["AC", "DC"], errors="ignore")
        .div(1e6)
    )  # TWh

elec_con_df = pd.concat([elec_con[scenario] for scenario in compare_scenarios], axis=1)
elec_con_df.columns = compare_scenarios
round(elec_con_df[elec_con_df.gt(1).any(axis=1)], 2)

In [ ]:
elec_con_df.filter(like="heat", axis=0).sum()
# much more heat consumption from urban central resistive heater but less capa in LowFlex

### heat

In [ ]:
# heat generation
heat_gen = {}
for scenario in compare_scenarios:
    heat_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(
            bus_carrier=["urban decentral heat", "rural heat", "urban central heat"],
            **kwargs,
        )
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

heat_gen_df = pd.concat([heat_gen[scenario] for scenario in compare_scenarios], axis=1)
heat_gen_df.columns = compare_scenarios
round(heat_gen_df[heat_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# heat consumption
heat_gen = {}
for scenario in compare_scenarios:
    heat_gen[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(
            bus_carrier=["urban decentral heat", "rural heat", "urban central heat"],
            **kwargs,
        )
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

heat_con_df = pd.concat([heat_gen[scenario] for scenario in compare_scenarios], axis=1)
heat_con_df.columns = compare_scenarios
round(heat_con_df[heat_con_df.gt(1).any(axis=1)], 2)

In [ ]:
# heat storage
heat_stores_c = [
    "urban central water tanks",
    "urban central water pits",
    "urban decentral water tanks",
    "urban decentral water pits",
    "rural water tanks",
]

fig, ax = plt.subplots(figsize=(10, 6))
for n, name in zip(networks.values(), networks.keys()):
    n = n[year]
    stores_i = n.stores[
        (n.stores.carrier.isin(heat_stores_c)) & (n.stores.bus.str.contains("DE"))
    ].index
    n.stores.loc[stores_i].e_nom_opt.sum() / 1e6  # TWh
    (n.stores_t.e[stores_i].sum(axis=1) / 1e6).plot(label=name)  # TWh
    plt.xlabel("Time [h]")
    plt.ylabel("Energy [TWh]")
    plt.title(f"Heat Storage - {year}")
    plt.legend()

### h2

In [ ]:
# hydrogen generation
h2_gen = {}
for scenario in compare_scenarios:
    h2_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["H2"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

h2_gen_df = pd.concat([h2_gen[scenario] for scenario in compare_scenarios], axis=1)
h2_gen_df.columns = compare_scenarios
round(h2_gen_df[h2_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# hydrogen consumption
h2_con = {}
for scenario in compare_scenarios:
    h2_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["H2"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

h2_con_df = pd.concat([h2_con[scenario] for scenario in compare_scenarios], axis=1)
h2_con_df.columns = compare_scenarios
round(h2_con_df[h2_con_df.gt(1).any(axis=1)], 2)

In [ ]:
# h2 storage
fig, ax = plt.subplots(figsize=(10, 6))
for n, name in zip(networks.values(), networks.keys()):
    n = n[year]
    stores_i = n.stores[
        (n.stores.carrier == "H2 Store") & (n.stores.bus.str.contains("DE"))
    ].index
    n.stores.loc[stores_i].e_nom_opt.sum() / 1e6  # TWh
    (n.stores_t.e[stores_i].sum(axis=1) / 1e6).plot(label=name)  # TWh
    plt.xlabel("Time [h]")
    plt.ylabel("Energy [TWh]")
    plt.title(f"Hydrogen Storage - {year}")
    plt.legend()

In [ ]:
round(results_by_country["H2"], 2)

### oil

In [ ]:
year = 2045

In [ ]:
# oil generation
oil_gen = {}
for scenario in compare_scenarios:
    oil_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["oil"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

oil_gen_df = pd.concat([oil_gen[scenario] for scenario in compare_scenarios], axis=1)
oil_gen_df.columns = compare_scenarios
oil_gen_df[oil_gen_df.gt(1).any(axis=1)]

In [ ]:
# oil consumption
oil_con = {}
for scenario in compare_scenarios:
    oil_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["oil"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

oil_con_df = pd.concat([oil_con[scenario] for scenario in compare_scenarios], axis=1)
oil_con_df.columns = compare_scenarios
oil_con_df[oil_con_df.gt(1).any(axis=1)]

In [ ]:
# renewable oil production
re_oil_gen = {}
for scenario in compare_scenarios:
    re_oil_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["renewable oil"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

re_oil_gen_df = pd.concat(
    [re_oil_gen[scenario] for scenario in compare_scenarios], axis=1
)
re_oil_gen_df.columns = compare_scenarios
round(re_oil_gen_df[re_oil_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# renewable oil consumption
re_oil_con = {}
for scenario in compare_scenarios:
    re_oil_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["renewable oil"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

re_oil_con_df = pd.concat(
    [re_oil_con[scenario] for scenario in compare_scenarios], axis=1
)
re_oil_con_df.columns = compare_scenarios
re_oil_con_df[re_oil_con_df.gt(1).any(axis=1)]

In [ ]:
# renewable oil imports

In [ ]:
# oil storage
fig, ax = plt.subplots(figsize=(10, 6))
for n, name in zip(networks.values(), networks.keys()):
    n = n[year]
    stores_i = n.stores[
        (n.stores.carrier == "oil") & (n.stores.bus.str.contains("DE"))
    ].index
    n.stores.loc[stores_i].e_nom_opt.sum() / 1e6  # TWh
    (n.stores_t.e[stores_i].sum(axis=1) / 1e6).plot(label=name)  # TWh
    plt.xlabel("Time [h]")
    plt.ylabel("Energy [TWh]")
    plt.title(f"Oil Storage - {year}")
    plt.legend()

### gas

In [ ]:
year = 2045

In [ ]:
# gas generation
gas_gen = {}
for scenario in compare_scenarios:
    gas_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["gas"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

gas_gen_df = pd.concat([gas_gen[scenario] for scenario in compare_scenarios], axis=1)
gas_gen_df.columns = compare_scenarios
round(gas_gen_df[gas_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# renewable gas generation
re_gas_gen = {}
for scenario in compare_scenarios:
    re_gas_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["renewable gas"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

re_gas_gen_df = pd.concat(
    [re_gas_gen[scenario] for scenario in compare_scenarios], axis=1
)
re_gas_gen_df.columns = compare_scenarios
round(re_gas_gen_df[re_gas_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# gas consumption
gas_con = {}
for scenario in compare_scenarios:
    gas_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["gas"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # TWh

gas_con_df = pd.concat([gas_con[scenario] for scenario in compare_scenarios], axis=1)
gas_con_df.columns = compare_scenarios
round(gas_con_df[gas_con_df.gt(1).any(axis=1)], 2)

In [ ]:
# gas storage
fig, ax = plt.subplots(figsize=(10, 6))
for n, name in zip(networks.values(), networks.keys()):
    n = n[year]
    stores_i = n.stores[
        (n.stores.carrier == "gas") & (n.stores.bus.str.contains("DE"))
    ].index
    n.stores.loc[stores_i].e_nom_opt.sum() / 1e6  # TWh
    (n.stores_t.e[stores_i].sum(axis=1) / 1e6).plot(label=name)  # TWh
    plt.xlabel("Time [h]")
    plt.ylabel("Energy [TWh]")
    plt.title(f"Gas Storage - {year}")
    plt.legend()

### co2

In [ ]:
year = 2045

In [ ]:
kwargs = {
    "groupby": pypsa.statistics.groupers["name", "bus", "carrier"],
    "at_port": True,
    "nice_names": False,
}

In [ ]:
# supply of CO2
co2_gen = {}
for scenario in compare_scenarios:
    co2_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["co2"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # Mio t

co2_gen_df = pd.concat([co2_gen[scenario] for scenario in compare_scenarios], axis=1)
co2_gen_df.columns = compare_scenarios
round(co2_gen_df[co2_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
co2_gen_df.sum()

In [ ]:
# withdrawal of CO2
co2_con = {}
for scenario in compare_scenarios:
    co2_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["co2"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # Mio t

co2_con_df = pd.concat([co2_con[scenario] for scenario in compare_scenarios], axis=1)
co2_con_df.columns = compare_scenarios
round(co2_con_df[co2_con_df.gt(1).any(axis=1)], 2)

### biomass

In [ ]:
# supply of biomass
biomass_gen = {}
for scenario in compare_scenarios:
    biomass_gen[scenario] = (
        networks[scenario][year]
        .statistics.supply(bus_carrier=["solid biomass"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # Mio t

biomass_gen_df = pd.concat(
    [biomass_gen[scenario] for scenario in compare_scenarios], axis=1
)
biomass_gen_df.columns = compare_scenarios
round(biomass_gen_df[biomass_gen_df.gt(1).any(axis=1)], 2)

In [ ]:
# withdrawal of biomass
biomass_con = {}
for scenario in compare_scenarios:
    biomass_con[scenario] = (
        networks[scenario][year]
        .statistics.withdrawal(bus_carrier=["solid biomass"], **kwargs)
        .filter(like="DE")
        .groupby(["carrier"])
        .sum()
        .div(1e6)
    )  # Mio t

biomass_con_df = pd.concat(
    [biomass_con[scenario] for scenario in compare_scenarios], axis=1
)
biomass_con_df.columns = compare_scenarios
round(biomass_con_df[biomass_con_df.gt(1).any(axis=1)], 2)

In [ ]:
# Biomass Trade

for n, name in zip(networks.values(), networks.keys()):
    n = n[year]

    biomass_potential_DE = (
        n.stores.query("carrier.str.contains('solid biomass')")
        .filter(like=region, axis=0)
        .e_nom.sum()
    )

    biomass_usage_local = (
        n.stores_t.p[
            n.stores.query("carrier.str.contains('solid biomass')")
            .filter(like=region, axis=0)
            .index
        ]
        .sum()
        .multiply(n.snapshot_weightings["stores"].unique().item())
        .sum()
    )

    biomass_usage_transported = (
        n.generators_t.p[
            n.generators.query("carrier.str.contains('solid biomass')")
            .filter(like=region, axis=0)
            .index
        ]
        .sum()
        .multiply(n.snapshot_weightings["generators"].unique().item())
        .sum()
    )

    biomass_net_exports = (
        biomass_potential_DE - biomass_usage_local - biomass_usage_transported
    )

    print(
        f"{name} biomass net exports: {round(biomass_net_exports / 1e6, 2)} TWh"
    )  # negative means imports

## curtailment

In [ ]:
# curtailment

for year in years:
    for n, name in zip(networks.values(), networks.keys()):
        n = n[year]
        carriers = [
            "solar",
            "solar rooftop",
            "solar-hsat",
            "offwind-ac",
            "offwind-dc",
            "onwind",
        ]
        print()
        print(f"Year: {year} - Scenario: {name}")
        for c in carriers:
            if c in n.generators.carrier.unique():
                ind = n.generators[n.generators.carrier == c].index
                gen_theoretical = (
                    n.generators_t.p_max_pu[ind] * n.generators.p_nom_opt[ind] * 3
                )
                gen_real = n.generators_t.p[ind] * 3
                curt = gen_theoretical - gen_real
                print(
                    f"Curtailment for {c}: {round(curt.values.sum() / 1e6, 2)} TWh (theoretical generation: {round(gen_theoretical.values.sum() / 1e6, 2)}); real generation: {round(gen_real.values.sum() / 1e6, 2)} TWh; curtailment percentage: {curt.values.sum() / gen_theoretical.values.sum() * 100:.2f} %"
                )

## energy balance

In [ ]:
# energy balances

balances = defaultdict(dict)
for year in years:
    for scenario in compare_scenarios:
        network = networks[scenario][year]
        ct = "DE"
        buses = network.buses.index[(network.buses.index.str[:2] == ct)].drop("DE")
        balance = (
            network.statistics.energy_balance(
                aggregate_time=False,
                nice_names=False,
                groupby=["bus", "carrier", "bus_carrier"],
            )
            .loc[:, buses, :, :]
            .droplevel("bus")
        )
        balances[scenario][year] = balance

### elec

In [ ]:
year = 2045

In [ ]:
carriers = ["AC", "low voltage"]

for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

### h2

In [ ]:
# h2 balance
carriers = ["H2"]

for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

### heat

In [ ]:
# heat balance
carriers = ["urban central heat", "rural heat", "urban decentral heat"]

for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

### co2

In [ ]:
# co2 balance
carriers = ["co2 stored"]
for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

### oil

In [ ]:
# oil balance
year = 2045
tech_colors["oil"] = "yellow"

carriers = ["oil"]
for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

In [ ]:
# renewable oil balance
carriers = ["renewable oil"]
for scenario in compare_scenarios:
    mask = balances[scenario][year].index.get_level_values("bus_carrier").isin(carriers)
    nb = balances[scenario][year][mask].groupby("carrier").sum().div(1e3).T
    plot_balance(
        nb,
        title=f"Energy Balance of - '{', '.join(carriers)}' ({scenario}, {year})",
        tech_colors=tech_colors,
    )

# More capacities

In [ ]:
# capacity heat

region = "DE"
xlims = {
    2020: (-100, 100),
    2025: (-100, 100),
    2030: (-100, 100),
    2035: (-200, 200),
    2040: (-200, 200),
    2045: (-200, 200),
}

for i, year in enumerate(years):
    fig, ax = plt.subplots(1, 1, figsize=(10, 15))

    bc = ["urban central heat", "rural heat", "urban decentral heat"]

    capas = {}
    for scenario in compare_scenarios:
        capas[scenario] = (
            networks[scenario][year]
            .statistics.optimal_capacity(bus_carrier=bc, **kwargs)
            .filter(like=region)
            .groupby("carrier")
            .sum()
            .div(1e3)
        )

    df = pd.concat([capas[scenario] for scenario in compare_scenarios], axis=1)
    df.columns = compare_scenarios
    df.plot(
        kind="barh",
        title=f"Optimal capacity in {year} ({bc})",
        xlim=xlims[year],
        ylabel="GW",
        ax=ax,
    )

    plt.grid()
    plt.tight_layout()

In [ ]:
# capacity h2

# capacity heat

region = "DE"
xlims = {
    2020: (-50, 50),
    2025: (-100, 100),
    2030: (-100, 100),
    2035: (-200, 200),
    2040: (-200, 200),
    2045: (-200, 200),
}

for i, year in enumerate(years):
    fig, ax = plt.subplots(1, 1, figsize=(10, 15))

    bc = ["H2"]

    capas = {}
    for scenario in compare_scenarios:
        capas[scenario] = (
            networks[scenario][year]
            .statistics.optimal_capacity(bus_carrier=bc, **kwargs)
            .filter(like=region)
            .groupby("carrier")
            .sum()
            .div(1e3)
        )

    df = pd.concat([capas[scenario] for scenario in compare_scenarios], axis=1)
    df.columns = compare_scenarios
    df.plot(
        kind="barh",
        title=f"Optimal capacity in {year} ({bc})",
        xlim=xlims[year],
        ylabel="GW",
        ax=ax,
    )

    plt.grid()
    plt.tight_layout()

# Prices

In [ ]:
carriers = ["AC", "low voltage"]

for year in years:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))

    for scenario in compare_scenarios:
        buses = networks[scenario][year].buses.index[
            (networks[scenario][year].buses.index.str[:2] == "DE")
            & (networks[scenario][year].buses.carrier.isin(carriers))
        ]
        prices = networks[scenario][year].buses_t.marginal_price[buses]

        print(f"\n{scenario} - {year} Marginal Prices Statistics:")
        print(prices.mean(axis=1).describe())
        print()

        # plot prices
        ax.plot(prices.mean(axis=1), label=f"{scenario} - {year}", ls=":")

        # plot pdc
        sorted_prices = np.sort(prices.mean(axis=1).values)
        pdc = np.arange(1, len(sorted_prices) + 1) / len(sorted_prices)
        ax2.plot(pdc, sorted_prices, label=f"{scenario} - {year}", ls=":")

    # Configure time series plot
    ax.set_title(f"Marginal Prices in {year} ({', '.join(carriers)})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Price (EUR/MWh)")
    ax.legend()
    ax.grid()
    ax.set_ylim(-50, 400)
    fig.tight_layout()

    # Configure price duration curve plot
    ax2.set_title(f"Price Duration Curve in {year} ({', '.join(carriers)})")
    ax2.set_xlabel("PDC")
    ax2.set_ylabel("Price (EUR/MWh)")
    ax2.legend()
    ax2.grid()
    ax2.set_ylim(-50, 400)
    fig2.tight_layout()

In [ ]:
carriers = ["urban central heat", "rural heat", "urban decentral heat"]

for year in years:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))

    for scenario in compare_scenarios:
        buses = networks[scenario][year].buses.index[
            (networks[scenario][year].buses.index.str[:2] == "DE")
            & (networks[scenario][year].buses.carrier.isin(carriers))
        ]
        prices = networks[scenario][year].buses_t.marginal_price[buses]

        print(f"\n{scenario} - {year} Marginal Prices Statistics:")
        print(prices.mean(axis=1).describe())
        print()

        # plot prices
        ax.plot(prices.mean(axis=1), label=f"{scenario} - {year}", ls=":")

        # plot pdc
        sorted_prices = np.sort(prices.mean(axis=1).values)
        pdc = np.arange(1, len(sorted_prices) + 1) / len(sorted_prices)
        ax2.plot(pdc, sorted_prices, label=f"{scenario} - {year}", ls=":")

    # Configure time series plot
    ax.set_title(f"Marginal Prices in {year} ({', '.join(carriers)})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Price (EUR/MWh)")
    ax.legend()
    ax.grid()
    ax.set_ylim(-50, 400)
    fig.tight_layout()

    # Configure price duration curve plot
    ax2.set_title(f"Price Duration Curve in {year} ({', '.join(carriers)})")
    ax2.set_xlabel("PDC")
    ax2.set_ylabel("Price (EUR/MWh)")
    ax2.legend()
    ax2.grid()
    ax2.set_ylim(-50, 400)
    fig2.tight_layout()

In [ ]:
carriers = ["H2"]

for year in years:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))

    for scenario in compare_scenarios:
        buses = networks[scenario][year].buses.index[
            (networks[scenario][year].buses.index.str[:2] == "DE")
            & (networks[scenario][year].buses.carrier.isin(carriers))
        ]
        prices = networks[scenario][year].buses_t.marginal_price[buses]

        print(f"\n{scenario} - {year} Marginal Prices Statistics:")
        print(prices.mean(axis=1).describe())
        print()

        # plot prices
        ax.plot(prices.mean(axis=1), label=f"{scenario} - {year}", ls=":")

        # plot pdc
        sorted_prices = np.sort(prices.mean(axis=1).values)
        pdc = np.arange(1, len(sorted_prices) + 1) / len(sorted_prices)
        ax2.plot(pdc, sorted_prices, label=f"{scenario} - {year}", ls=":")

    # Configure time series plot
    ax.set_title(f"Marginal Prices in {year} ({', '.join(carriers)})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Price (EUR/MWh)")
    ax.legend()
    ax.grid()
    ax.set_ylim(-50, 400)
    fig.tight_layout()

    # Configure price duration curve plot
    ax2.set_title(f"Price Duration Curve in {year} ({', '.join(carriers)})")
    ax2.set_xlabel("PDC")
    ax2.set_ylabel("Price (EUR/MWh)")
    ax2.legend()
    ax2.grid()
    ax2.set_ylim(-50, 400)
    fig2.tight_layout()

In [ ]:
assert 0

# Testing

In [ ]:
n = networks["MedFlex"][2045]

In [ ]:
n.stores.carrier.unique()

In [ ]:
pth_links = n.links.index[
    (
        (
            n.links.carrier.str.contains("rural")
            | n.links.carrier.str.contains("decentral")
        )
        & (
            n.links.carrier.str.contains("heat pump")
            | n.links.carrier.str.contains("resistive heater")
        )
    )
    & ~n.links.carrier.str.contains("urban central")  # exclude central systems
    & n.links.bus0.str.contains("DE")  #
]

In [ ]:
pth_links

## Carbon removal

In [ ]:
n_1cl_3H_voll_old[2025].global_constraints

In [ ]:
n_1cl_3H_voll_new[2025].global_constraints

In [ ]:
# co2 supply
# co2 withdrawal
n = n_1cl_3H_voll_new[2030]
co2_gen = round(n.statistics.supply(bus_carrier="co2") / 1e6, 2)  # TWh
co2_gen[co2_gen > 10]

In [ ]:
# co2 withdrawal
n = n_1cl_3H_voll_new[2030]
co2_con = round(n.statistics.withdrawal(bus_carrier="co2") / 1e6, 2)  # TWh
co2_con[co2_con > 10]

In [ ]:
n.links[n.links.carrier == "DAC"].marginal_cost

In [ ]:
# co2 removal service
n = n_1cl_3H_voll_new[2045]
n.generators_t.p["CO2 removal service"]

In [ ]:
year = 2045
dir = "/home/julian-geis/repos/pypsa-de-pricing/results/20250414-Pricing-Co2Removal/"
run = f"KN2045_Bal_v4_voll/networks/base_s_1__none_{year}_lt.nc"
n = pypsa.Network(dir + run)

In [ ]:
n.generators[n.generators.carrier == "CO2 removal service"][
    ["bus", "efficiency", "p_nom_opt", "p_min_pu", "p_max_pu", "marginal_cost", "sign"]
]

In [ ]:
n.generators_t.p["CO2 removal service"].plot()

In [ ]:
# co2 consumption
n = n_1cl_3H_voll_new[2045]
co2_con = round(n.statistics.withdrawal(bus_carrier="co2") / 1e6, 2)  # TWh
co2_con[co2_con > 10]

In [ ]:
co2_gen = round(n.statistics.supply(bus_carrier="co2") / 1e6, 2)  # TWh
co2_gen[co2_gen > 10]

In [ ]:
# amount of removed co2
n = n_1cl_3H_voll_new[2045]
n.generators_t.p["CO2 removal service"].sum() / 1e6 * 3  # 3 Mio tonnes

In [ ]:
# costs of co2 removal
2.707404e06 / 1e6  # 2.7 Mio EUR

In [ ]:
co2_bal = n.statistics.energy_balance(
    bus_carrier="co2",
    aggregate_time=False,
    nice_names=False,
    groupby=["bus", "carrier", "bus_carrier"],
).droplevel(["component", "bus", "bus_carrier"])

In [ ]:
co2_bal.loc["DAC"].plot()
co2_bal.loc["CO2 removal service"].plot()

## prices

In [ ]:
n = n_1cl_3H_voll_new[2045]

n.buses_t.marginal_price["DE renewable gas"].plot(label="DE renewable gas")
n.buses_t.marginal_price["EU renewable gas"].plot(label="EU renewable gas")
n.buses_t.marginal_price["DE renewable oil"].plot(label="DE renewable oil")
n.buses_t.marginal_price["EU renewable oil"].plot(label="EU renewable oil")
plt.legend()

In [ ]:
# add co2 costs
specific_emissions = {
    "oil": 0.2571,
    "oil primary": 0.2571,
    "gas": 0.198,  # OCGT
    "gas primary": 0.198,  # OCGT
    "coal": 0.3361,
    "lignite": 0.4069,
}

n.global_constraints

In [ ]:
n.generators.loc["Renewable oil import"]

## energy balance plots

In [ ]:
import matplotlib.dates as mdates


def plot_balance_custom(nb, tech_colors=None, year=None):
    resample = "D"
    nb = nb.resample(resample).mean()

    df = nb

    # split into df with positive and negative values
    df_neg, df_pos = df.clip(upper=0), df.clip(lower=0)
    df_pos = df_pos[df_pos.sum().sort_values(ascending=False).index]
    df_neg = df_neg[df_neg.sum().sort_values().index]
    # get colors
    c_neg = [
        tech_colors[col] if col in tech_colors else "grey" for col in df_neg.columns
    ]
    c_pos = [
        tech_colors[col] if col in tech_colors else "grey" for col in df_pos.columns
    ]

    fig, ax = plt.subplots(figsize=(14, 5))

    # plot positive values
    ax = df_pos.plot.area(ax=ax, stacked=True, color=c_pos, linewidth=0.0)

    # rename negative values that are also present on positive side, so that they are not shown and plot negative values
    def f(c):
        return "out_" + c

    cols = [f(c) if (c in df_pos.columns) else c for c in df_neg.columns]
    cols_map = dict(zip(df_neg.columns, cols))
    ax = df_neg.rename(columns=cols_map).plot.area(
        ax=ax, stacked=True, color=c_neg, linewidth=0.0
    )

    # explicitly filter out duplicate labels
    handles, labels = ax.get_legend_handles_labels()
    filtered_handles_labels = [
        (h, l) for h, l in zip(handles, labels) if not l.startswith("out_")
    ]
    handles, labels = zip(*filtered_handles_labels)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    # rescale the y-axis
    ax.set_ylim([1.05 * df_neg.sum(axis=1).min(), 1.05 * df_pos.sum(axis=1).max()])
    ax.legend(
        handles,
        labels,
        ncol=2,
        loc="upper center",
        bbox_to_anchor=(1.27, 1.01),
    )
    if year != "":
        ax.text(
            0.02,
            0.95,
            str(year),
            horizontalalignment="left",
            verticalalignment="top",
            transform=ax.transAxes,
            fontsize=18,
            color="black",
            bbox=dict(
                facecolor="white", edgecolor="none", alpha=0.8, boxstyle="round,pad=0.3"
            ),
        )
    ax.set_ylabel("GW", fontsize=16)
    ax.set_xlabel("")
    ax.grid(True)

    return fig

In [ ]:
# elec generation and consumption

for year in np.arange(2020, 2050, 5):
    n = n_1cl_3H_voll_new[year]
    balance = n.statistics.energy_balance(
        aggregate_time=False,
        nice_names=False,
        groupby=["bus", "carrier", "bus_carrier"],
    ).droplevel("bus")

    carriers = ["AC"]
    mask = balance.index.get_level_values("bus_carrier").isin(carriers)
    nb = balance[mask].groupby("carrier").sum().div(1e3).T

    fig = plot_balance_custom(nb, tech_colors)
    fig.savefig(f"{PLOT_DIR}/electricity-balance-plain-{year}.png", bbox_inches="tight")

In [ ]:
# elec generation and consumption

for year in np.arange(2020, 2050, 5):
    n = n_1cl_3H_voll_new[year]
    balance = n.statistics.energy_balance(
        aggregate_time=False,
        nice_names=False,
        groupby=["bus", "carrier", "bus_carrier"],
    ).droplevel("bus")

    carriers = ["AC"]
    mask = balance.index.get_level_values("bus_carrier").isin(carriers)
    nb = balance[mask].groupby("carrier").sum().div(1e3).T
    nb = get_condense_sum(nb, c1_groups, c1_groups_name)
    nb = nb.rename(columns=carrier_renaming)

    fig = plot_balance_custom(nb, tech_colors, year)
    fig.savefig(f"{PLOT_DIR}/electricity-balance-{year}.png", bbox_inches="tight")

# capacity plots

In [ ]:
cap_all = pd.DataFrame()

for year in years:
    n = n_1cl_3H_voll_new[year]

    cap = (
        n.statistics.optimal_capacity(bus_carrier=["AC"], nice_names=False)
        .div(1e3)  # MW → GW
        .droplevel(0)
        .to_frame(name=year)
    )

    cap_all = cap_all.combine_first(cap) if not cap_all.empty else cap

cap_all = cap_all.drop("load-shedding", errors="ignore")

# aggragate technologies
# exclude solar
if "solar" in c1_groups_name:
    i = c1_groups_name.index("solar")
    del c1_groups[i]
    del c1_groups_name[i]

df_new = cap_all.copy()

all_grouped_rows = []

for group, name in zip(c1_groups, c1_groups_name):
    existing = [device for device in group if device in df_new.index]
    if existing:
        # Avoid name conflict: only sum if the group name isn't among the grouped rows
        df_new.loc[name] = df_new.loc[existing].sum()
        # Exclude the group name itself from deletion (e.g. 'solar')
        all_grouped_rows.extend([d for d in existing if d != name])

# Drop rows that were part of groupings (except where name conflicts)
cap_all_agg = df_new.drop(index=all_grouped_rows)

techs = cap_all_agg.abs().sum(axis=1).sort_values(ascending=False).head(16).index
cap_agg_top = cap_all_agg.loc[techs]

In [ ]:
# Make a copy to avoid mutating the original df
df_plot = cap_agg_top.copy()

# Fill NaNs with 0 for plotting
df_plot = df_plot.fillna(0)

# Determine ordering based on absolute capacity in 2045
sorted_techs = df_plot[2045].abs().sort_values(ascending=False).index.tolist()

# Years to plot
years = [2020, 2025, 2030, 2035, 2040, 2045]

# Prepare plot
fig, axes = plt.subplots(3, 2, figsize=(14, 14), sharex=False, sharey=True)

# Iterate over each year
for i, year in enumerate(years):
    row, col = i // 2, i % 2
    ax = axes[row, col]

    # Get data for this year, ordered
    year_data = df_plot[year].loc[sorted_techs]

    # Separate supply and demand
    supply = year_data[year_data >= 0]
    demand = -year_data[year_data < 0]  # make demand positive for plotting

    # Positions: supply on left, demand on right
    pos_supply = np.arange(len(supply))
    pos_demand = np.arange(len(demand)) + len(supply) + 1  # +1 gap

    # Plot bars
    ax.bar(
        pos_supply,
        supply.values,
        color=[tech_colors.get(tech, "#1f77b4") for tech in supply.index],
        alpha=0.8,
    )
    ax.bar(
        pos_demand,
        demand.values,
        color=[tech_colors.get(tech, "#d62728") for tech in demand.index],
        alpha=0.8,
    )

    # Divider line between supply and demand
    divider_x = len(supply) - 0.5
    ax.axvline(x=divider_x, color="black", linestyle="--", linewidth=1.0)

    # Add 'Supply' and 'Demand' labels using axis coordinates
    ax.text(
        0.02,
        0.95,
        "Supply",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold",
    )

    ax.text(
        0.98,
        0.95,
        "Demand",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=10,
        fontweight="bold",
    )

    # Bar labels
    for x, y in zip(pos_supply, supply.values):
        ax.text(
            x,
            y + max(supply.max(), demand.max()) * 0.01,
            f"{y:.0f}",
            ha="center",
            va="bottom",
            fontsize=6,
        )
    for x, y in zip(pos_demand, demand.values):
        ax.text(
            x,
            y + max(supply.max(), demand.max()) * 0.01,
            f"{y:.0f}",
            ha="center",
            va="bottom",
            fontsize=6,
        )

    # X-axis labels
    tech_labels = list(supply.index) + list(demand.index)
    all_positions = list(pos_supply) + list(pos_demand)
    ax.set_xticks(all_positions)
    ax.set_xticklabels(tech_labels, rotation=45, ha="right", fontsize=8)

    # Title and labels
    ax.set_title(f"{year}", fontsize=16)
    ax.set_ylabel("GW", fontsize=12)
    ax.grid(True, axis="y", alpha=0.3)
    ax.axhline(y=0, color="black", linewidth=0.8)

# Layout and save
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/electricity-capacity.png", bbox_inches="tight")

# costs

In [ ]:
def calculate_annuity(n: float, r: float | pd.Series) -> float | pd.Series:
    """
    Calculate the annuity factor for an asset with lifetime n years and discount rate r.

    The annuity factor is used to calculate the annual payment required to pay off a loan
    over n years at interest rate r. For example, annuity(20, 0.05) * 20 = 1.6.

    Parameters
    ----------
    n : float
        Lifetime of the asset in years
    r : float | pd.Series
        Discount rate (interest rate). Can be a single float or a pandas Series of rates.

    Returns
    -------
    float | pd.Series
        Annuity factor. Returns a float if r is float, or pd.Series if r is pd.Series.

    Examples
    --------
    >>> calculate_annuity(20, 0.05)
    0.08024258718774728
    """
    if isinstance(r, pd.Series):
        return pd.Series(1 / n, index=r.index).where(
            r == 0, r / (1.0 - 1.0 / (1.0 + r) ** n)
        )
    elif r > 0:
        return r / (1.0 - 1.0 / (1.0 + r) ** n)
    else:
        return 1 / n


def prepare_costs(cost_file, params, nyears):
    for key in ("marginal_cost", "capital_cost"):
        if key in params:
            params["overwrites"][key] = params[key]

    # set all asset costs and other parameters
    costs = pd.read_csv(cost_file, index_col=[0, 1]).sort_index()

    # correct units to MW and EUR
    costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3

    # min_count=1 is important to generate NaNs which are then filled by fillna
    costs = (
        costs.loc[:, "value"].unstack(level=1).groupby("technology").sum(min_count=1)
    )

    costs = costs.fillna(params["fill_values"])

    for attr in ("investment", "lifetime", "FOM", "VOM", "efficiency", "fuel"):
        overwrites = params["overwrites"].get(attr)
        if overwrites is not None:
            overwrites = pd.Series(overwrites)
            costs.loc[overwrites.index, attr] = overwrites

    def annuity_factor(v):
        return calculate_annuity(v["lifetime"], v["discount rate"]) + v["FOM"] / 100

    costs["capital_cost"] = [
        annuity_factor(v) * v["investment"] * nyears for i, v in costs.iterrows()
    ]

    for attr, key in dict(marginal_cost="marginal_cost", capital_cost="fixed").items():
        overwrites = params["overwrites"].get(attr)
        if overwrites is not None:
            overwrites = pd.Series(overwrites)
            idx = overwrites.index.intersection(costs.index)
            costs.loc[idx, key] = overwrites

    return costs


def load_costs(tech_costs, config, max_hours, Nyears=1.0):
    for key in ("marginal_cost", "capital_cost"):
        if key in config:
            config["overwrites"][key] = config[key]

    # set all asset costs and other parameters
    costs = pd.read_csv(tech_costs, index_col=[0, 1]).sort_index()

    # correct units from kW to MW
    costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
    costs.unit = costs.unit.str.replace("/kW", "/MW")

    # correct units from GW to MW
    costs.loc[costs.unit.str.contains("/GW"), "value"] /= 1e3
    costs.unit = costs.unit.str.replace("/GW", "/MW")

    fill_values = config["fill_values"]
    costs = costs.value.unstack().fillna(fill_values)

    for attr in ("investment", "lifetime", "FOM", "VOM", "efficiency", "fuel"):
        overwrites = config["overwrites"].get(attr)
        if overwrites is not None:
            overwrites = pd.Series(overwrites)
            costs.loc[overwrites.index, attr] = overwrites

    costs["capital_cost"] = (
        (
            calculate_annuity(costs["lifetime"], costs["discount rate"])
            + costs["FOM"] / 100.0
        )
        * costs["investment"]
        * Nyears
    )
    costs.at["OCGT", "fuel"] = costs.at["gas", "fuel"]
    costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]

    costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]

    costs = costs.rename(columns={"CO2 intensity": "co2_emissions"})

    costs.at["OCGT", "co2_emissions"] = costs.at["gas", "co2_emissions"]
    costs.at["CCGT", "co2_emissions"] = costs.at["gas", "co2_emissions"]

    costs.at["solar", "capital_cost"] = costs.at["solar-utility", "capital_cost"]
    costs.at["solar", "investment"] = costs.at["solar-utility", "investment"]

    costs = costs.rename({"solar-utility single-axis tracking": "solar-hsat"})

    def costs_for_storage(store, link1, link2=None, max_hours=1.0):
        capital_cost = link1["capital_cost"] + max_hours * store["capital_cost"]
        overnight_cost = link1["investment"] + max_hours * store["investment"]
        if link2 is not None:
            capital_cost += link2["capital_cost"]
            overnight_cost += link2["investment"]
        return pd.Series(
            dict(
                capital_cost=capital_cost,
                overnight_cost=overnight_cost,
                marginal_cost=0.0,
                co2_emissions=0.0,
            )
        )

    costs.loc["battery"] = costs_for_storage(
        costs.loc["battery storage"],
        costs.loc["battery inverter"],
        max_hours=max_hours["battery"],
    )
    costs.loc["H2"] = costs_for_storage(
        costs.loc["hydrogen storage underground"],
        costs.loc["fuel cell"],
        costs.loc["electrolysis"],
        max_hours=max_hours["H2"],
    )

    for attr in ("marginal_cost", "capital_cost", "overnight_cost"):
        overwrites = config["overwrites"].get(attr)
        if overwrites is not None:
            overwrites = pd.Series(overwrites)
            costs.loc[overwrites.index, attr] = overwrites

    return costs


params = {
    "year": 2030,
    "version": "v0.10.1",
    "social_discountrate": 0.02,
    "fill_values": {
        "FOM": 0,
        "VOM": 0,
        "efficiency": 1,
        "fuel": 0,
        "investment": 0,
        "lifetime": 25,
        "CO2 intensity": 0,
        "discount rate": 0.07,
    },
    "overwrites": {},
    "marginal_cost": {
        "solar": 0.01,
        "onwind": 0.015,
        "offwind": 0.015,
        "hydro": 0.0,
        "H2": 0.0,
        "electrolysis": 0.0,
        "fuel cell": 0.0,
        "battery": 0.0,
        "battery inverter": 0.0,
        "home battery storage": 0,
        "water tank charger": 0.03,
    },
    "emission_prices": {"enable": False, "co2": 0.0, "co2_monthly_prices": False},
    "horizon": "mean",
    "NEP": 2021,
    "transmission": "overhead",
}

max_hours = {"battery": 6, "H2": 168}

# costs = prepare_costs(
#     costs_input,
#     params,
#     1,
# )

# costs_loaded = load_costs(
#     costs_input,
#     params,
#     max_hours,
#     1,
# )

In [ ]:
run = "20250603-Pricing-Methanol"

fuels = ["oil", "coal", "lignite", "gas", "solid biomass", "methanol"]
years = [2020, 2025, 2030, 2035, 2040, 2045]
res = pd.DataFrame(index=fuels, columns=years)

for year in years:
    costs_input = f"/home/julian-geis/repos/pypsa-de-pricing/resources/{run}/KN2045_Bal_v4_voll/costs_{year}.csv"

    costs_loaded = load_costs(
        costs_input,
        params,
        max_hours,
        1,
    )
    for fuel in fuels:
        res.loc[fuel, year] = costs_loaded.loc[fuel, "fuel"]

res.loc["solid biomass import"] = res.loc["solid biomass"] + 0.1333 * 200

In [ ]:
res.round(0)

In [ ]:
c = "lignite"
year = 2020
n_1cl_3H_voll_new[year].generators[
    n_1cl_3H_voll_new[year].generators.carrier.str.contains(c)
][["bus", "p_nom_opt", "marginal_cost"]]

In [ ]:
assert 0

## Plots

## Playground

In [ ]:
n = n_ariadne[2045]

In [ ]:
n.lines[n.lines.bus0.str.contains("DE")]

In [ ]:
n.links[(n.links.carrier == "DC") & (n.links.bus0.str.contains("DE"))]

In [ ]:
# Helper: checks if a bus name starts with 'DE'
def is_de(bus_name):
    return bus_name.str.startswith("DE")


# Filter lines
n.lines = n.lines[
    ~(
        is_de(n.lines.bus0) & is_de(n.lines.bus1)  # both ends in DE
        | is_de(n.lines.bus0) & ~is_de(n.lines.bus1)  # DE to other
        | ~is_de(n.lines.bus0) & is_de(n.lines.bus1)  # other to DE
    )
]

# Filter DC links (assuming carrier is 'DC' or similar)
ac_links = n.links.carrier.str.contains("DC", case=False, na=False)
de_links = is_de(n.links.bus0) | is_de(n.links.bus1)

n.links = n.links[~(ac_links & de_links)]

In [ ]:
n.links[n.links.carrier == "DC"]

In [ ]:
(
    n.stores[n.stores.carrier.str.startswith("co2 sequestered")].bus.map(
        n.buses.location
    )
    == "EU"
)

In [ ]:
carrier

In [ ]:
n.global_constraints

In [ ]:
n.stores[n.stores.carrier == "co2 sequestered"][["e_nom_opt"]].sum()

In [ ]:
(
    n.stores[n.stores.carrier == "co2 sequestered"].bus.map(n.buses.location).unique()
    == "EU"
).all()

In [ ]:
n.generators[n.generators.carrier.str.contains("solid biomas")]